In [ ]:
!pip install nltk -q
!pip install nltk scikit-learn -q
!pip install contractions

In [ ]:
!pip install spacy nltk nlpaug
!python -m spacy download en_core_web_sm

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from sklearn.feature_extraction.text import TfidfVectorizer
import contractions
import re
import nltk
import spacy
import nlpaug.augmenter.word as naw
from nltk.corpus import stopwords
import pandas as pd


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import kagglehub

# 1. Download dataset
path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")
print("Path to dataset files:", path)

# 2. Find the downloaded CSV file path
csv_file = os.path.join(path, "WELFake_Dataset.csv")

# 3. Read into pandas DataFrame
df = pd.read_csv(csv_file)

# 4. Preview the data
print(df.head())
print(df.info())

In [ ]:
import numpy as np
import pandas as pd


def remove_garbage(df):
    df_clean = df.copy()

    # 1. Drop index column if present (e.g., 'Unnamed: 0')
    if "Unnamed: 0" in df_clean.columns:
        df_clean = df_clean.drop(columns=["Unnamed: 0"])

    # 2. Remove exact duplicate rows across all columns
    df_clean = df_clean.drop_duplicates()

    # 3. Drop rows where BOTH 'title' and 'text' are missing (completely empty articles)
    if "title" in df_clean.columns and "text" in df_clean.columns:
        df_clean = df_clean.dropna(subset=["title", "text"], how="all")

    # 4. Fill remaining single-column NaNs with empty strings
    # (e.g., if an article has a valid body text but a missing title, don't throw away the row)
    df_clean["title"] = df_clean["title"].fillna("")
    df_clean["text"] = df_clean["text"].fillna("")

    # 5. Combine title and text to create a complete feature text column
    df_clean["content"] = (
        df_clean["title"].str.strip() + " " + df_clean["text"].str.strip()
    )

    # 6. Remove rows where combined 'content' is just whitespace or empty
    df_clean = df_clean[df_clean["content"].str.strip() != ""]

    # 7. Drop remaining duplicates based purely on the combined content text to prevent overfitting
    df_clean = df_clean.drop_duplicates(subset=["content"])

    # 8. Drop single-value (zero-variance) columns
    zero_var_cols = [col for col in df_clean.columns if df_clean[col].nunique() <= 1]
    df_clean = df_clean.drop(columns=zero_var_cols)

    # 9. Drop high-cardinality ID columns (excluding target variable 'label')
    id_like_cols = [
        col
        for col in df_clean.select_dtypes(include=["object", "int"]).columns
        if col != "label" and df_clean[col].nunique() == len(df_clean)
    ]
    df_clean = df_clean.drop(columns=id_like_cols)

    return df_clean


# Execute cleaning
df_clean = remove_garbage(df)

# Sanity Check
print("Remaining NaN values:\n", df_clean.isna().sum())
print(f"Cleaned dataset shape: {df_clean.shape}")

In [ ]:
nltk.download('stopwords')
nlp = spacy.load("en_core_web_sm")

In [ ]:
!pip install spacy nlpaug
!python -m spacy download en_core_web_sm

In [ ]:
import os
import pandas as pd
import spacy

# 1. Load spaCy model with unneeded pipeline components disabled
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])


# 2. Define process function for a single batch of text strings
def process_batch_spacy(texts_batch):
    cleaned_texts = []
    # spaCy nlp.pipe streams the texts efficiently
    for doc in nlp.pipe(texts_batch, batch_size=500, n_process=1):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if token.is_alpha and not token.is_stop
        ]
        cleaned_texts.append(" ".join(tokens))
    return cleaned_texts


# 3. Process CSV Chunk-by-Chunk
input_csv = (
    "WELFake_Dataset.csv"  # Ensure this points to your file path or kaggle path
)
output_csv = "WELFake_Cleaned.csv"

# Remove old output file if re-running
if os.path.exists(output_csv):
    os.remove(output_csv)

CHUNK_SIZE = 5000  # Process 5,000 rows at a time to stay well under RAM limits
total_processed = 0

print(
    f"Processing '{input_csv}' in chunks of {CHUNK_SIZE} rows to save RAM...\n"
)

# pd.read_csv with chunksize creates an iterator instead of loading everything into RAM
for chunk_idx, chunk in enumerate(
    pd.read_csv(input_csv, chunksize=CHUNK_SIZE)
):
    # Combine title & text
    chunk["title"] = chunk["title"].fillna("")
    chunk["text"] = chunk["text"].fillna("")
    content_series = (chunk["title"] + " " + chunk["text"]).astype(str)

    # Process batch
    chunk["processed_text"] = process_batch_spacy(content_series.tolist())

    # Keep only the essential columns to keep the saved CSV small
    keep_cols = [
        col for col in ["label", "processed_text"] if col in chunk.columns
    ]
    chunk_to_save = chunk[keep_cols]

    # Append to output CSV file on disk
    write_header = not os.path.exists(output_csv)
    chunk_to_save.to_csv(output_csv, mode="a", index=False, header=write_header)

    total_processed += len(chunk)
    print(f"Processed batch {chunk_idx + 1}: {total_processed} total rows saved.")

print("\n--- Processing Complete! ---")
# Load a small sample of the processed data to verify
df_sample = pd.read_csv(output_csv, nrows=5)
print(df_sample)

In [ ]:
import pandas as pd

# 1. Load the cleaned output CSV generated by your previous script
df_all = pd.read_csv("WELFake_Cleaned.csv")

# 2. Map 'processed_text' to 'text' so your inspection code works seamlessly
df_all["text"] = df_all["processed_text"].fillna("").astype(str)
df_all["label"] = df_all["label"].astype(int)

# Optional: verify shape and missing values
print(f"Successfully loaded df_all with shape: {df_all.shape}")

In [ ]:
# Look at the first ~40 characters of a sample from each class
print(df_all[df_all['label']==1]['text'].str[:60].sample(15).tolist())
print(df_all[df_all['label']==0]['text'].str[:60].sample(15).tolist())

In [ ]:
import re

def strip_dateline(text):
    # Reuters-style: "WASHINGTON (Reuters) - actual article starts here"
    return re.sub(r'^[A-Z][A-Za-z\s]+\(Reuters\)\s*-\s*', '', text)

df_all['text'] = df_all['text'].apply(strip_dateline)

In [ ]:
stop_words = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

In [ ]:
def wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [ ]:
def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    tagged = pos_tag(tokens)
    lemmas = [lemmatizer.lemmatize(word, wordnet_pos(tag)) for word, tag in tagged]
    return ' '.join(lemmas)

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 256  # paired title + article; raise if longer context is needed
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Use df_all, the labeled dataset assembled above. Keep title and article as a pair.
# df_all['title'] = df_all['title'].fillna('').astype(str)
df_all['text'] = df_all['text'].fillna('').astype(str)
df_all['label'] = df_all['label'].astype(int)


In [ ]:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# import torch

# MODEL_DIR = "trained_model"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_DIR,
#     num_labels=2
# )

# # Freeze the full BERT encoder
# for param in model.bert.parameters():
#     param.requires_grad = False

# # Unfreeze only the final two Transformer blocks
# for layer in model.bert.encoder.layer[-2:]:
#     for param in layer.parameters():
#         param.requires_grad = True

# # Keep classification head trainable
# for param in model.classifier.parameters():
#     param.requires_grad = True
# device =  torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model.to(device)

# print(
#     'Trainable parameters:',
#     sum(p.numel() for p in model.parameters() if p.requires_grad)
# )

# print(
#     'Total parameters:',
#     sum(p.numel() for p in model.parameters())
# )

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    classifier_dropout = 0.3
)

# Freeze the full BERT encoder
for param in model.bert.parameters():
    param.requires_grad = False

# Unfreeze only the final two Transformer blocks
for layer in model.bert.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# Keep classification head trainable
for param in model.classifier.parameters():
    param.requires_grad = True

device =  torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(
    'Trainable parameters:',
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

print(
    'Total parameters:',
    sum(p.numel() for p in model.parameters())
)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# 1. Custom PyTorch Dataset for Hugging Face Tokenizer
class NewsDataset(Dataset):
    # def __init__(self, texts, titles, labels, tokenizer, max_len=256):
    #     self.texts = texts
    #     self.titles = titles
    #     self.labels = labels
    #     self.tokenizer = tokenizer
    #     self.max_len = max_len
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Tokenize title and body text paired together
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        return item, torch.tensor(label, dtype=torch.long)

# 2. Split dataframe into training and validation sets
train_df, val_df = train_test_split(
    df_all,
    test_size=0.2,
    random_state=42,
    stratify=df_all['label']
)

# 3. Create Dataset instances
train_dataset = NewsDataset(
    texts=train_df['text'].tolist(),
    labels=train_df['label'].tolist(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = NewsDataset(
    texts=val_df['text'].tolist(),
    labels=val_df['label'].tolist(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# 4. Define batch size and create DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau


criterion = nn.CrossEntropyLoss()

EPOCHS = 5
LEARNING_RATE = 2e-5

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=0.02,
)

# total_steps = len(train_loader) * EPOCHS
#  scheduler = get_linear_schedule_with_warmup(
#      optimizer,
#      num_warmup_steps=int(total_steps * 0.10),  # 10% warmup
#      num_training_steps=total_steps,
#  )

best_val_loss = float("inf")
print("starting...")
for epoch in range(EPOCHS):

    # ==================== TRAIN ====================

    model.train()
    running_loss = 0.0

    for batch, labels in train_loader:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch, labels=labels)
        logits = outputs.logits

        loss = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            1.0
        )

        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / max(1, len(train_loader))

    print(
        f'Epoch {epoch + 1}/{EPOCHS} - '
        f'Train Loss: {avg_train_loss:.4f}'
    )

    # ==================== VALIDATION ====================

    model.eval()

    all_preds = []
    all_labels = []
    val_loss = 0.0

    with torch.no_grad():

        for batch, labels in val_loader:

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            labels = labels.to(device)

            logits = model(**batch).logits

            loss = criterion(logits, labels)

            val_loss += loss.item()

            all_preds.extend(
                logits.argmax(dim=1).cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    avg_val_loss = val_loss / len(val_loader)

    #scheduler.step(avg_val_loss)

    print(f'Validation Loss: {avg_val_loss:.4f}')

    print(
        f'Validation Accuracy: '
        f'{accuracy_score(all_labels, all_preds):.4f}'
    )

    print(
        classification_report(
            all_labels,
            all_preds,
            target_names=['fake', 'real'],
            zero_division=0
        )
    )

    # ==================== SAVE BEST MODEL ====================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        OUTPUT_DIR = 'trained_model'

        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

        print(
            f'New best validation loss: '
            f'{best_val_loss:.4f}. '
            f'Saved model to {OUTPUT_DIR}'
        )

In [ ]:
!pip install lime
from lime.lime_text import LimeTextExplainer
explainer=LimeTextExplainer(class_names= ["real","fake"])
no_of_words=5

In [ ]:
def explanation (text,model,no_of_words=5):
  #in the method explain_instance the inputs are the text
  #then the models prediction on changing each word in sentence
  #last one is how many of the highest probabilty words that
  #we want to have


  reasoning = explainer.explain_instance(
    text,
    model.predict_proba,
    num_features = min(no_of_words,len(text.strip().split("")))
  )
  list=[]
  for word,weight in  reasoning.as_list():
    list.append(
        {
            "word":word,
            "weight":weight,
            "prediction":  "towards real" if weight>0 else "towards fake"
        }
      )
  return list

In [ ]:
def analyze (text,model,no_of_words=5):
    if not text or not text.strip():
        return {
            "prediction": None,
            "confidence": None,
            "explanation": [],
            "error": "no text provided",
        }
        no_of_words = min(no_of_words,len(text.strip().split("")))
        prediction=model.predict([text])[0]
        confidence=float(max(model.predict_proba([text])[0].max()))
        return { "prediction":prediction,
              "confidence":confidence,
              "explanation":explanation(text,model,no_of_words)
              }

In [ ]:
pip install streamlit

import joblib
joblib.dump(model,"model.pkl")
joblib.dump(tokenizer,"tokenizer.pkl")
